# 01 — Reconhecimento do Portal ANEEL

**Objetivo:** inspecionar o portal `www2.aneel.gov.br/cedoc/` antes de escrever qualquer código de scraping.

**Por que fazer isso primeiro?**  
Scraping é altamente dependente da estrutura HTML de cada site. Se você escreve o `scraper.py` sem entender o site, vai reescrever várias vezes. Uma hora aqui economiza dias depois.

**O que vamos descobrir neste notebook:**
1. O servidor aceita nossas requisições? (status HTTP, headers)
2. Qual é a estrutura HTML da tabela de índice das RENs?
3. Quais metadados estão disponíveis? (número, ementa, situação, data)
4. Como funciona a paginação?
5. As URLs dos PDFs seguem o padrão documentado?
6. O conteúdo é HTML estático ou carregado via JavaScript?

---
**Contexto do projeto:** as respostas aqui vão diretamente para `src/ingestion/scraper.py` e `DECISIONS.md`.

## 0. Setup — instalar bibliotecas no Colab

In [ ]:
# No Colab, requests e lxml já costumam estar instalados.
# Instalamos explicitamente para garantir versões corretas.
!pip install requests beautifulsoup4 lxml -q

import requests
from bs4 import BeautifulSoup
import time
import json

print('Bibliotecas carregadas com sucesso.')

## 1. Primeira requisição — o portal responde?

Antes de qualquer parsing, precisamos saber:
- O servidor retorna status 200 (OK)?
- Há redirecionamentos?
- Qual o encoding da resposta? (importante para acentuação em português)
- O servidor bloqueia bots? (User-Agent padrão do requests às vezes é bloqueado)

In [ ]:
# URL base do portal de documentos da ANEEL
BASE_URL = 'https://www2.aneel.gov.br/cedoc/'

# Cabeçalho HTTP que simula um navegador comum.
# Alguns servidores bloqueiam requisições sem User-Agent ou com o padrão do requests.
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'pt-BR,pt;q=0.9',
}

response = requests.get(BASE_URL, headers=HEADERS, timeout=30)

print(f'Status HTTP:    {response.status_code}')
print(f'URL final:      {response.url}')  # mostra se houve redirecionamento
print(f'Encoding:       {response.encoding}')
print(f'Content-Type:   {response.headers.get("Content-Type", "não informado")}')
print(f'Tamanho (bytes):{len(response.content)}')
print()

# Forçamos o encoding correto para português
# Se response.encoding estiver errado, os acentos vão aparecer como caracteres estranhos
if response.encoding and response.encoding.lower() not in ('utf-8', 'utf8'):
    print(f'⚠️  Encoding não é UTF-8. Caracteres especiais podem precisar de tratamento.')

## 2. Inspecionar o HTML bruto

Antes de usar BeautifulSoup, vale olhar um trecho do HTML cru para ter intuição do que vem do servidor.

In [ ]:
# Mostra os primeiros 3000 caracteres do HTML
# Suficiente para ver: DOCTYPE, encoding declarado, estrutura inicial
print(response.text[:3000])

In [ ]:
# Parseia o HTML com BeautifulSoup usando lxml como backend
# lxml é mais tolerante a HTML malformado (comum em portais governamentais)
soup = BeautifulSoup(response.text, 'lxml')

# Conta quantas tabelas existem na página
tabelas = soup.find_all('table')
print(f'Número de tabelas na página: {len(tabelas)}')

# Lista os IDs e classes de cada tabela para identificar a correta
for i, tabela in enumerate(tabelas):
    print(f'  Tabela {i}: id="{tabela.get("id", "")}" class="{tabela.get("class", "")}"')

## 3. Identificar a tabela de índice das RENs

O portal deve ter uma tabela listando as resoluções com seus metadados. Vamos identificar qual tabela é essa e quais são seus cabeçalhos (colunas).

In [ ]:
# Tenta encontrar todas as tabelas com linhas de dados
# Adaptação: mude o seletor se o portal usar div/ul em vez de table
for i, tabela in enumerate(tabelas):
    linhas = tabela.find_all('tr')
    if len(linhas) > 3:  # descarta tabelas de layout com poucas linhas
        print(f'--- Tabela {i} ({len(linhas)} linhas) ---')
        # Mostra o cabeçalho (primeira linha)
        cabecalho = linhas[0].find_all(['th', 'td'])
        colunas = [c.get_text(strip=True) for c in cabecalho]
        print(f'Colunas: {colunas}')
        print()

In [ ]:
# -----------------------------------------------------------------------
# AJUSTE AQUI: substitua o índice abaixo pelo número da tabela correta
# que contém as RENs (descoberto na célula anterior)
# -----------------------------------------------------------------------
INDICE_TABELA = 0  # <- mude este número conforme necessário

tabela_rens = tabelas[INDICE_TABELA]
linhas = tabela_rens.find_all('tr')

print(f'Tabela selecionada tem {len(linhas)} linhas (incluindo cabeçalho).')
print()
print('=== Primeiras 3 linhas de dados (HTML bruto) ===')
for linha in linhas[1:4]:  # pula cabeçalho, mostra 3 exemplos
    print(linha)
    print()

## 4. Extrair metadados de uma REN

Com a tabela identificada, vamos extrair os dados de uma linha para entender o schema real disponível no portal — número, ementa, situação, data, URL do PDF.

In [ ]:
def extrair_linha(linha):
    """
    Extrai todos os campos de uma linha da tabela do portal.
    Retorna um dict com os dados encontrados.
    
    Esta função será o núcleo do scraper.py — aqui você descobre
    os seletores CSS/atributos corretos para cada campo.
    """
    celulas = linha.find_all('td')
    if not celulas:
        return None
    
    resultado = {}
    
    # Mostra o conteúdo de cada célula com seu índice
    for i, celula in enumerate(celulas):
        texto = celula.get_text(strip=True)
        links = [a.get('href', '') for a in celula.find_all('a')]
        resultado[f'celula_{i}'] = {'texto': texto, 'links': links}
    
    return resultado

# Testa com a 2ª linha (índice 1 = primeira linha de dados, pulando cabeçalho)
primeira_ren = extrair_linha(linhas[1])
print(json.dumps(primeira_ren, ensure_ascii=False, indent=2))

In [ ]:
# -----------------------------------------------------------------------
# Com base no output acima, mapeie os índices das células:
#
# Exemplo (AJUSTE conforme o que você viu):
#   celula_0 → número da resolução
#   celula_1 → ano
#   celula_2 → ementa
#   celula_3 → situação (vigente/revogada)
#   celula_4 → data de publicação
#   celula_X (com link) → URL do PDF
# -----------------------------------------------------------------------

# Preencha depois de analisar o output anterior:
MAPA_COLUNAS = {
    'numero':           None,  # ex.: 0
    'ano':              None,  # ex.: 1
    'ementa':           None,  # ex.: 2
    'situacao':         None,  # ex.: 3
    'data_publicacao':  None,  # ex.: 4
    'coluna_pdf':       None,  # ex.: 5 (a célula que tem o link para o PDF)
}

print('Mapa de colunas definido (edite as linhas acima com os índices corretos):')
print(json.dumps(MAPA_COLUNAS, ensure_ascii=False, indent=2))

## 5. Investigar paginação

O portal provavelmente não carrega todas as RENs numa única página. Precisamos entender como navegar pelas páginas — via parâmetro de URL, via formulário POST, ou outro mecanismo.

In [ ]:
# Procura por links/botões de paginação no HTML
# Padrões comuns: 'próximo', 'next', 'page', 'página', '>>'
termos_paginacao = ['próximo', 'proximo', 'next', 'anterior', 'page', 'pag', '>>']

links = soup.find_all('a')
print(f'Total de links na página: {len(links)}')
print()

print('Links que parecem ser de paginação:')
for link in links:
    texto = link.get_text(strip=True).lower()
    href = link.get('href', '')
    if any(t in texto for t in termos_paginacao) or any(t in href.lower() for t in termos_paginacao):
        print(f'  texto="{link.get_text(strip=True)}" href="{href}"')

In [ ]:
# Procura por formulários — alguns portais governamentais usam POST para paginação
formularios = soup.find_all('form')
print(f'Formulários na página: {len(formularios)}')

for i, form in enumerate(formularios):
    action = form.get('action', '')
    method = form.get('method', 'GET').upper()
    inputs = [(inp.get('name', ''), inp.get('value', '')) for inp in form.find_all('input')]
    print(f'  Form {i}: method={method} action="{action}"')
    print(f'    Campos: {inputs}')

In [ ]:
# Tenta parâmetros comuns de paginação via GET
# Muitos sistemas usam ?pagina=2, ?page=2, ?offset=20, etc.

params_paginacao_candidatos = [
    {'pagina': 2},
    {'page': 2},
    {'pg': 2},
    {'start': 20},
    {'offset': 20},
]

print('Testando parâmetros de paginação...')
for params in params_paginacao_candidatos:
    try:
        r = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)
        # Compara tamanho da resposta com a página 1
        # Se for diferente, algo mudou — pode ser paginação funcionando
        mesmo_tamanho = abs(len(r.content) - len(response.content)) < 500
        print(f'  params={params} → status={r.status_code}, '
              f'tamanho={len(r.content)} bytes '
              f'({"igual à pág 1" if mesmo_tamanho else "DIFERENTE da pág 1 ← investigar"})')
        time.sleep(1)  # respeito ao servidor
    except Exception as e:
        print(f'  params={params} → erro: {e}')

## 6. Verificar padrão de URL dos PDFs

O `CLAUDE.md` documenta o padrão esperado:
```
https://www2.aneel.gov.br/cedoc/ren{ano}{numero:04d}.pdf
```

Vamos verificar se isso está correto com algumas RENs conhecidas.

In [ ]:
# RENs conhecidas para validar o padrão de URL
# Fonte: dataset JvPetas/aneel-legislacao no HuggingFace (referência do projeto)
rens_teste = [
    {'ano': 2021, 'numero': 1000},  # REN 1000/2021
    {'ano': 2010, 'numero': 414},   # REN 414/2010
    {'ano': 2005, 'numero': 165},   # REN 165/2005
    {'ano': 2000, 'numero': 57},    # REN 57/2000 (era antiga)
]

def montar_url_pdf(ano: int, numero: int) -> str:
    """Monta a URL do PDF conforme o padrão documentado."""
    return f'https://www2.aneel.gov.br/cedoc/ren{ano}{numero:04d}.pdf'

def montar_url_consolidada(ano: int, numero: int) -> str:
    """Monta a URL da versão consolidada (prefixo 'bren')."""
    return f'https://www2.aneel.gov.br/cedoc/bren{ano}{numero:04d}.pdf'

print('Verificando acessibilidade dos PDFs...\n')
for ren in rens_teste:
    url = montar_url_pdf(ren['ano'], ren['numero'])
    url_cons = montar_url_consolidada(ren['ano'], ren['numero'])
    
    try:
        # HEAD request — verifica se existe sem baixar o arquivo inteiro
        r = requests.head(url, headers=HEADERS, timeout=15, allow_redirects=True)
        r_cons = requests.head(url_cons, headers=HEADERS, timeout=15, allow_redirects=True)
        
        tamanho = r.headers.get('Content-Length', '?')
        print(f'REN {ren["numero"]}/{ren["ano"]}:')
        print(f'  Original:    {r.status_code} — {url}')
        print(f'  Consolidada: {r_cons.status_code} — {url_cons}')
        if tamanho != '?':
            print(f'  Tamanho PDF: {int(tamanho) / 1024:.0f} KB')
        print()
        time.sleep(1)
    except Exception as e:
        print(f'REN {ren["numero"]}/{ren["ano"]}: erro — {e}\n')

## 7. Testar extração de texto de um PDF

Antes de implementar o `extractor.py`, vale verificar se o PyMuPDF consegue extrair texto de uma REN típica — e se PDFs antigos são digitais ou escaneados.

In [ ]:
!pip install PyMuPDF -q
import fitz  # PyMuPDF
import io

def testar_extracao_pdf(url: str, nome: str) -> None:
    """
    Baixa um PDF em memória e testa a extração de texto.
    NUNCA salva o PDF em disco — apenas processa em memória.
    Isso é a premissa do projeto: dados não tocam a máquina.
    """
    print(f'=== {nome} ===')
    try:
        r = requests.get(url, headers=HEADERS, timeout=60)
        if r.status_code != 200:
            print(f'  Erro HTTP: {r.status_code}\n')
            return
        
        # Abre o PDF direto do bytes em memória (sem salvar em disco)
        pdf = fitz.open(stream=io.BytesIO(r.content), filetype='pdf')
        num_paginas = len(pdf)
        
        # Extrai texto da primeira página
        pagina_0 = pdf[0]
        texto = pagina_0.get_text()
        
        print(f'  Páginas:       {num_paginas}')
        print(f'  Chars pág. 1:  {len(texto)}')
        print(f'  Texto extraído? {"SIM (PDF digital)" if len(texto) > 100 else "NÃO (provavelmente escaneado)"}')
        print(f'  Primeiros 300 chars:')
        print(f'  {texto[:300].strip()}')
        pdf.close()
    except Exception as e:
        print(f'  Erro: {e}')
    print()

# Testa REN recente (esperamos PDF digital)
testar_extracao_pdf(
    montar_url_pdf(2021, 1000),
    'REN 1000/2021 (recente)'
)

time.sleep(2)

# Testa REN antiga (risco de ser escaneada)
testar_extracao_pdf(
    montar_url_pdf(2000, 57),
    'REN 57/2000 (antiga)'
)

## 8. Registro de achados — preencha após executar o notebook

**Este é o output mais importante do notebook.** Copie as respostas para `DECISIONS.md`.

In [ ]:
# Preencha este dicionário com os achados do reconhecimento
# Será a base do scraper.py

ACHADOS = {
    # --- Servidor ---
    'status_http': None,                  # ex.: 200
    'encoding': None,                     # ex.: 'utf-8' ou 'iso-8859-1'
    'bloqueou_requisicao': None,          # True / False

    # --- Estrutura HTML ---
    'tipo_conteudo': None,                # 'html_estatico' ou 'javascript'
    'seletor_tabela': None,               # ex.: 'table#gridView' ou tabelas[0]
    'num_colunas': None,                  # quantas colunas tem a tabela
    'mapa_colunas': MAPA_COLUNAS,         # preenchido na seção 4

    # --- Paginação ---
    'mecanismo_paginacao': None,          # 'GET ?param=N', 'POST', 'sem_paginacao'
    'parametro_paginacao': None,          # ex.: 'pagina' ou 'page'
    'itens_por_pagina': None,             # ex.: 20
    'total_rens_estimado': None,          # se o portal informar

    # --- URLs dos PDFs ---
    'padrao_url_valido': None,            # True / False
    'urls_com_404': [],                   # lista de RENs que não responderam

    # --- Extração de texto ---
    'pdfs_recentes_digitais': None,       # True / False
    'pdfs_antigos_escaneados': None,      # True / False — se sim, OCR necessário
    'ano_limite_ocr': None,               # ex.: 2003 (antes disso, provavelmente scanned)
}

print('=== ACHADOS DO RECONHECIMENTO ===')
print(json.dumps(ACHADOS, ensure_ascii=False, indent=2, default=str))

## Próximos passos

Com os achados registrados acima:

1. **Copie os achados para `DECISIONS.md`** com a data de hoje — eles explicam por que o `scraper.py` foi implementado da forma que foi.

2. **Atualize `PROGRESS.md`** — marque este notebook como concluído e desbloqueie `src/ingestion/scraper.py`.

3. **Implemente `src/ingestion/scraper.py`** — agora você tem:
   - O seletor correto da tabela
   - O mapeamento das colunas
   - O mecanismo de paginação
   - O padrão de URL validado

4. **Se PDFs antigos forem escaneados** — anote o `ano_limite_ocr` e implemente o fallback de OCR no `extractor.py`.

---
*Notebook criado em 2026-05-28 | Projeto: ANEEL RAG Benchmark*